# Credit-card fraud: H2O AutoML

Run each cell with `Shift+Enter`, top to bottom. The notebook reads `H2O.csv`,
the training set the **tfo-H2O** step writes, trains up to five models with
H2O AutoML, and ranks them.

In **JupyterLab**, start it from the lab folder so the notebook finds the data.
In **Colab**, run the two cells marked *Colab only* as well.

In [ ]:
# Colab only: install H2O (JupyterLab uses the h2o-env virtual environment)
# !pip install h2o -q

In [ ]:
# Import libraries
import os
from datetime import datetime

import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import matplotlib.pyplot as plt

# On the lab machine, start H2O on Pentaho's Java. H2O reads H2O_JAVA_HOME
# before any java on PATH.
pentaho_java = r"C:\Pentaho\java"
if os.path.isdir(pentaho_java):
    os.environ["H2O_JAVA_HOME"] = pentaho_java

h2o.init(max_mem_size="4G", nthreads=-1)
print("\nH2O initialized.")

In [ ]:
# Colab only: upload H2O.csv
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# Load data: the file the transformation wrote, else the Lab Files copy
path = os.path.join("data", "H2O.csv") if os.path.exists(os.path.join("data", "H2O.csv")) else "H2O.csv"
data = pd.read_csv(path, sep=";", header=None)

data.columns = [
    "first_time_customer", "order_dollar_amount", "num_items", "age",
    "web_order", "total_transactions", "hour_of_day",
    "billing_shipping_match", "fraud",
]

# Convert target to string
data["fraud"] = data["fraud"].astype(str)

print(f"Read {path}")
print(f"Dataset: {data.shape[0]:,} rows × {data.shape[1]} columns")
print(f"Fraud cases: {(data['fraud'] == '1').sum():,} ({(data['fraud'] == '1').mean() * 100:.2f}%)")

In [ ]:
# Quick class distribution check
fraud_dist = data["fraud"].value_counts().sort_index()

plt.figure(figsize=(10, 4))
fraud_dist.plot(kind="bar", color=["green", "red"])
plt.title("Class Distribution", fontweight="bold")
plt.xlabel("Class (0=Normal, 1=Fraud)")
plt.ylabel("Count")
plt.xticks([0, 1], ["Normal", "Fraud"], rotation=0)
for i, v in enumerate(fraud_dist):
    plt.text(i, v + 50, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Convert to H2O frame
hf = h2o.H2OFrame(data)
hf["fraud"] = hf["fraud"].asfactor()

# Split: 75% train, 25% test
train, test = hf.split_frame(ratios=[0.75], seed=42)

print(f"Training: {train.nrows:,} rows")
print(f"Test:     {test.nrows:,} rows")

# Define features and target
x = train.columns
x.remove("fraud")
y = "fraud"

In [ ]:
# Fast configuration
start_time = datetime.now()

aml = H2OAutoML(
    max_models=5,                   # Only 5 models for speed
    max_runtime_secs=900,           # 15 minutes max
    seed=42,
    balance_classes=True,           # Handle imbalanced data
    nfolds=5,                       # 5-fold CV
    sort_metric="AUCPR",
    include_algos=[
        "GBM",           # Gradient Boosting (usually best)
        "GLM",           # Generalized Linear Model (fast)
        "DRF",           # Distributed Random Forest (tree-based)
        "DeepLearning",  # Neural Network
        # "XGBoost",     # add on Linux or Colab: H2O's XGBoost does not run on Windows
    ],
    verbosity="warn",
)

# Train
aml.train(x=x, y=y, training_frame=train)

duration = (datetime.now() - start_time).total_seconds()
print(f"\nTraining completed in {duration / 60:.1f} minutes.")

In [ ]:
# View leaderboard
lb = aml.leaderboard
print(lb)
print(f"\nTotal models trained: {lb.nrows}")
print(f"Best model: {lb.as_data_frame()['model_id'][0]}")

In [ ]:
# Get best model and score it on the test rows
best = aml.leader
perf = best.model_performance(test)

# Precision, recall and F1 at the threshold that gives the best F1
threshold = perf.find_threshold_by_max_metric("f1")

print(f"BEST MODEL: {best.model_id}")
print(f"Algorithm: {best.algo}")
print()
print(f"AUC:       {perf.auc():.4f}")
print(f"AUCPR:     {perf.aucpr():.4f}")
print(f"Precision: {perf.precision(thresholds=[threshold])[0][1]:.4f}")
print(f"Recall:    {perf.recall(thresholds=[threshold])[0][1]:.4f}")
print(f"F1 Score:  {perf.F1(thresholds=[threshold])[0][1]:.4f}")

In [ ]:
# Confusion matrix, at the same threshold
cm = perf.confusion_matrix(thresholds=threshold)
print(cm)

cm_table = cm.table.as_data_frame()
tn, fp = int(cm_table.iloc[0, 1]), int(cm_table.iloc[0, 2])
fn, tp = int(cm_table.iloc[1, 1]), int(cm_table.iloc[1, 2])
print(f"True Negatives:  {tn:>6,}")
print(f"False Positives: {fp:>6,}")
print(f"False Negatives: {fn:>6,} (missed fraud)")
print(f"True Positives:  {tp:>6,}")
print(f"\nCaught {tp} fraud cases, missed {fn}.")

In [ ]:
# Top features
varimp = best.varimp(use_pandas=True)
print(varimp.head(10))

# Plot
best.varimp_plot()

In [ ]:
# Save the best model in automl_output, next to the notebook
model_path = h2o.save_model(model=best, path="automl_output", force=True)
print(f"Model saved: {model_path}")

# Colab only: download it
# files.download(model_path)

In [ ]:
# Generate summary
report = f"""
FRAUD DETECTION - QUICK SUMMARY
{'=' * 60}

Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Dataset:
  Total:     {len(data):,}
  Training:  {train.nrows:,}
  Test:      {test.nrows:,}

Model:
  Best:      {best.model_id}
  Algorithm: {best.algo}
  Training:  {duration / 60:.1f} minutes

Performance:
  AUC:       {perf.auc():.4f}
  AUCPR:     {perf.aucpr():.4f}
  Precision: {perf.precision(thresholds=[threshold])[0][1]:.4f}
  Recall:    {perf.recall(thresholds=[threshold])[0][1]:.4f}
  F1:        {perf.F1(thresholds=[threshold])[0][1]:.4f}

Model saved: {model_path}

{'=' * 60}
"""

print(report)

with open("fraud_summary.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("Summary saved to fraud_summary.txt.")

# Colab only: download it
# files.download("fraud_summary.txt")